# Temporal split

Issue: #6

## Question

How do we split the four weeks so that evaluation only ever uses events after the training data?

## Data used

Input: `data/sample/events.parquet` (from #4). Code: `notebooks/split.py` (`from split import split_by_time, split_window, WINDOWS`), which stays out of `src/ranking/` until #52.

Output: nothing saved. The function returns lazy train and eval frames for a window.

## The windows

Days count from 2022-07-31 22:00 UTC. Week 1 = days 0-6, week 2 = 7-13, week 3 = 14-20, week 4 = 21-27.

| Window | Train | Eval |
|---|---|---|
| w0 initial | days 0-13 (weeks 1-2) | days 14-15 |
| w1 week 3 arrives | days 0-20 (weeks 1-3) | days 21-22 |
| w2 week 4 arrives | days 0-25 (weeks 1-3 + 5 days of week 4) | days 26-27 |

Every evaluation window is the 2 days directly after its training data. The data ends on day 27, so week 4's last two days are held back for evaluation and cannot also be training data. The windows are defined once, in `WINDOWS`.

**Session rule.** Sessions can cross a boundary. Train keeps every event before `train_end` (a crossing session is cut there, so only its past is used). An eval session is one whose *first* event falls in the eval window, and its events are cut at `eval_end`. A session that started before the eval window is never an eval session, so no session is in both sets.

## Why a random split leaks

A random split puts some events of a session, and events from the same days, into both train and eval. The model then learns co-visitation, popularity and trends from events that happen *after* some of the events it is scored on. Real search only ever has the past, so the score would look better than it can be in use. Splitting by time removes that.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from split import DAY_MS, START_MS, WINDOWS, day_to_ms, split_by_time, split_window

DATA = Path("data/sample") if Path("data/sample").exists() else Path("../data/sample")

## A test with 10 hand-made events and a known answer

Boundaries: train_end = eval_start = day 14, eval_end = day 16.

| Session | Events (day) | Expected |
|---|---|---|
| A | 1, 2 | train only |
| B | 13.5, 14.5 | crosses the boundary: 13.5 in train, B is not an eval session |
| C | 14.2, 14.8 | eval only |
| D | 15.5, 16.5 | eval session, the day-16.5 event is cut |
| E | 17, 18 | neither |

In [ ]:
def d(day):
    return day_to_ms(day)


toy = pl.DataFrame(
    {
        "session": ["A", "A", "B", "B", "C", "C", "D", "D", "E", "E"],
        "aid": list(range(10)),
        "ts": [d(1), d(2), d(13.5), d(14.5), d(14.2), d(14.8), d(15.5), d(16.5), d(17), d(18)],
    }
)
train, evaluation = split_by_time(toy, d(14), d(14), d(16))
train, evaluation = train.collect(), evaluation.collect()

assert sorted(train["aid"].to_list()) == [0, 1, 2]          # A (2 events) + B's first event
assert sorted(evaluation["aid"].to_list()) == [4, 5, 6]     # C (2 events) + D's first event
assert set(evaluation["session"]) == {"C", "D"}
assert not set(train["session"]) & set(evaluation["session"]) - {"B"}  # only B could be in train
assert "B" not in set(evaluation["session"])
assert train["ts"].max() < evaluation["ts"].min()
print("toy example passes: train aids", sorted(train["aid"].to_list()), "eval aids", sorted(evaluation["aid"].to_list()))

# bad boundaries are rejected
try:
    split_by_time(toy, d(15), d(14), d(16))
except ValueError as err:
    print("ValueError:", err)
else:
    raise AssertionError

## Real data: assertions for every window

In [ ]:
events = pl.scan_parquet(DATA / "events.parquet")
rows = []
for name, w in WINDOWS.items():
    train, evaluation = split_window(events, name)
    t = train.select(
        pl.len().alias("train_events"),
        pl.col("session").n_unique().alias("train_sessions"),
        pl.col("ts").max().alias("train_max"),
    ).collect()
    e = evaluation.select(
        pl.len().alias("eval_events"),
        pl.col("session").n_unique().alias("eval_sessions"),
        pl.col("ts").min().alias("eval_min"),
        pl.col("ts").max().alias("eval_max"),
    ).collect()
    # 1. The last training event is before the first evaluation event.
    assert t["train_max"][0] < e["eval_min"][0], name
    # 2. Evaluation stays inside its window.
    assert e["eval_min"][0] >= day_to_ms(w["eval_start_day"]), name
    assert e["eval_max"][0] < day_to_ms(w["eval_end_day"]), name
    # 3. No session is in both sets.
    overlap = (
        train.select("session").unique().join(evaluation.select("session").unique(), on="session")
        .select(pl.len()).collect()[0, 0]
    )
    assert overlap == 0, (name, overlap)
    rows.append({"window": name, **t.drop("train_max").to_dicts()[0], **e.drop(["eval_min", "eval_max"]).to_dicts()[0]})
print(pl.DataFrame(rows))
print("all windows: train ends before eval, eval inside its window, no shared sessions")

## How many sessions cross the boundary?

These are cut in train and are not used as eval sessions, so it matters how many there are.

In [ ]:
rows = []
spans = events.group_by("session").agg(pl.col("ts").min().alias("first"), pl.col("ts").max().alias("last")).collect()
for name, w in WINDOWS.items():
    boundary = day_to_ms(w["train_end_day"])
    crossing = spans.filter((pl.col("first") < boundary) & (pl.col("last") >= boundary)).height
    rows.append({"window": name, "sessions_crossing_train_end": crossing, "share_of_all": crossing / spans.height})
print(pl.DataFrame(rows))

## The windows on a timeline

In [ ]:
per_day = (
    events.with_columns(((pl.col("ts") - START_MS) // DAY_MS).alias("day"))
    .group_by("day").agg(pl.len().alias("n")).sort("day").collect()
)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True, gridspec_kw={"height_ratios": [1, 1.4]})
ax1.bar(per_day["day"], per_day["n"], color="lightgrey")
ax1.set_ylabel("events per day")
for y, (name, w) in enumerate(WINDOWS.items()):
    ax2.barh(y, w["train_end_day"], left=0, color="tab:blue", label="train" if y == 0 else None)
    ax2.barh(y, w["eval_end_day"] - w["eval_start_day"], left=w["eval_start_day"], color="tab:orange", label="eval" if y == 0 else None)
ax2.set_yticks(range(len(WINDOWS)), list(WINDOWS))
ax2.invert_yaxis()
ax2.set_xlabel("day since 2022-07-31 22:00 UTC")
ax2.legend(loc="lower left")
plt.tight_layout()
plt.show()

## Result

The toy example gives the expected answer. On the real sample, for all three windows the last training event is before the first evaluation event, evaluation stays inside its two days, and no session appears in both sets. The timeline shows that for each window, train ends exactly where eval begins.

Things to carry forward:
- **Sessions are long.** 34 percent of all sessions cross the day-14 boundary (27 percent cross day 21, 13 percent cross day 26). OTTO sessions are not single visits; they can span days. Those sessions are cut in train and are not eval sessions, so evaluation uses only sessions that *start* inside the eval window: about 100,000 for w0, 68,000 for w1 and 62,000 for w2.
- Eval sessions are therefore biased towards short-lived sessions that begin in the two eval days. Say so wherever results are reported.
- Window w2 has less new training data than a full week 4 (its last two days are held out), and later windows re-use earlier weeks as training data, so results across windows are not independent.

## What I learned

To be written by the owner of the project.